# Neural Network RAT classification

In [1]:
import sys
sys.path.append("..")

from src.preprocessing import encode_values, convert_to_numeric
from src.utils_train_models import (load_preprocessed_dataset, 
                                    generate_outcome_training_per_z, 
                                    store_windowed_training_analysis)
from src.utils_eval import compute_statistics
from src.config import path_list
import pandas as pd
import numpy as np
import os

In [2]:
MODEL_NAME = "MLPClassifier"

#### Data preprocessing

The main purposes of the **load_preprocessed_dataset** function are:
- reading the preprocessed dataset, if any;
- computing the dataset for feature extraction;<br>

The most important aspect is the *build_single_source_pipeline* function, which provides a common input feed for training both NN and RandomForest models.<br>
Furthermore, this step is required in the final section of the project to gain a fair analysis and comparison of the outcome of the trained models.<br>
The pipeline is composed of a sequence of steps listed in src/preprocessing.py.<br>
Given the .csv files with all measurement data retrieved from the source mentioned in README.md, it concatenates information in a single dataset.<br>
Concatenation can be performed by joining common columns, more precisely by joining rows according to their source node information.<br>
A subsequent mandatory step is *dataset preprocessing and optimization*:
- NaN analysis and deletion of columns whose information loss rate is higher than the average computed among all columns;
- deletion of duplicated rows;
- remaining NaN values filled with 0.0;

In [ ]:
dataset = load_preprocessed_dataset()
 
print("Precomputed dataset:")
print(dataset)

##### Output: preprocessed dataset generation
```small_test
Generated metadata with fields: Index(["id", "run", "node_name", "location", "modem_name", "mcc", "country",
       "iso_code", "rat", "rat_name"],
      dtype="object")

Generated non metadata with fields: Index(["id", "run", "node_name", "location", "modem_name", "mcc", "country",
       "iso_code", "rat", "rat_name", "timestamp_pck", "target_ip_pck",
       "icmp_seq_pck", "ttl_pck", "rtt_ms_pck", "timestamp_pck_loss",
       "icmp_seq_pck_loss", "ttl_pck_loss", "rtt_ms_pck_loss",
       "timestamp_throughput", "tot_size_throughput", "avg_speed_throughput",
       "tot_time_throughput", "filesize_throughput", "direction_throughput",
       "timestamp_current", "tot_size_current", "avg_speed_current",
       "tot_time_current", "filesize_current", "direction_current",
       "timestamp_pw_idle", "timetamp_ms_pw_idle", "diff_pw_idle",
       "current_pw_idle", "voltage_pw_idle", "timestamp_pw_upload",
       "avg_speed_pw_upload", "tot_time_pw_upload", "timetamp_ms_pw_upload",
       "diff_pw_upload", "current_pw_upload", "voltage_pw_upload"],
      dtype="object")

Average loss computed: 54.86

                 name_col   tot_row  info_lost  valid_info  percentage_loss
0           timestamp_pck  63587384   51392119    12195265            80.82
1           target_ip_pck  63587384   51392119    12195265            80.82
2            icmp_seq_pck  63587384   51392119    12195265            80.82
3                 ttl_pck  63587384   51392119    12195265            80.82
4              rtt_ms_pck  63587384   51392119    12195265            80.82
5      timestamp_pck_loss  63587384   53110235    10477149            83.52
6       icmp_seq_pck_loss  63587384   53110235    10477149            83.52
7            ttl_pck_loss  63587384   53110235    10477149            83.52
8         rtt_ms_pck_loss  63587384   53110235    10477149            83.52
9    timestamp_throughput  63587384   14688946    48898438            23.10
10    tot_size_throughput  63587384   14688946    48898438            23.10
11   avg_speed_throughput  63587384   14688946    48898438            23.10
12    tot_time_throughput  63587384   14688946    48898438            23.10
13    filesize_throughput  63587384   14688946    48898438            23.10
14   direction_throughput  63587384   14688946    48898438            23.10
15      timestamp_current  63587384   14688946    48898438            23.10
16       tot_size_current  63587384   14688946    48898438            23.10
17      avg_speed_current  63587384   14688946    48898438            23.10
18       tot_time_current  63587384   14688946    48898438            23.10
19       filesize_current  63587384   14688946    48898438            23.10
20      direction_current  63587384   14688946    48898438            23.10
21      timestamp_pw_idle  63587384   12532592    51054792            19.71
22    timetamp_ms_pw_idle  63587384   12532592    51054792            19.71
23           diff_pw_idle  63587384   12532592    51054792            19.71
24        current_pw_idle  63587384   12532592    51054792            19.71
25        voltage_pw_idle  63587384   12532592    51054792            19.71
26    timestamp_pw_upload  63587384   63252870      334514            99.47
27    avg_speed_pw_upload  63587384   63252868      334516            99.47
28     tot_time_pw_upload  63587384   63252868      334516            99.47
29  timetamp_ms_pw_upload  63587384   63252870      334514            99.47
30         diff_pw_upload  63587384   63252870      334514            99.47
31      current_pw_upload  63587384   63252870      334514            99.47
32      voltage_pw_upload  63587384   63252870      334514            99.47
nan values analysis stored in /Users/username/NDA-Lab-Project4-RAT-Classification-TL/data/analysis/nan_analysis.txt

Dropped 16 columns with above-average NaN rate
Preprocessed dataset stored at: /Users/username/NDA-Lab-Project4-RAT-Classification-TL/data/outcome_preprocess/feature_dataset.csv

Precomputed dataset:
             id  run node_name           location    modem_name  mcc  country  \
0         17443    1   Mark-10    Zagreb, Croatia  Quectel_BG96  219  Croatia   
10        17558    1   Mark-10    Zagreb, Croatia  Quectel_EC21  219  Croatia   
57        17760    1   Mark-10    Zagreb, Croatia  Quectel_BG96  219  Croatia   
105       17768    1   Mark-10    Zagreb, Croatia  Quectel_EC21  219  Croatia   
2864      18830    1   Mark-10    Zagreb, Croatia  Quectel_EC21  219  Croatia   
...         ...  ...       ...                ...           ...  ...      ...   
17136562  23995    1    Mark-6    Munich, Germany  Quectel_BG96  262  Germany   
17140707  24001    1    Mark-1  Würzburg, Germany  Quectel_BG96  262  Germany   
17145467  24003    1    Mark-6    Munich, Germany  Quectel_BG96  262  Germany   
17149035  24009    1    Mark-1  Würzburg, Germany  Quectel_BG96  262  Germany   
17153702  24011    1    Mark-6    Munich, Germany  Quectel_BG96  262  Germany   

         iso_code  rat rat_name  ...  tot_size_current  avg_speed_current  \
0              hr    0       2G  ...               0.0                0.0   
10             hr    0       2G  ...               0.0                0.0   
57             hr    0       2G  ...               0.0                0.0   
105            hr    0       2G  ...               0.0                0.0   
2864           hr    0       2G  ...               0.0                0.0   
...           ...  ...      ...  ...               ...                ...   
17136562       de    9   NB-IoT  ...               0.0                0.0   
17140707       de    9   NB-IoT  ...               0.0                0.0   
17145467       de    9   NB-IoT  ...               0.0                0.0   
17149035       de    9   NB-IoT  ...               0.0                0.0   
17153702       de    9   NB-IoT  ...               0.0                0.0   

          tot_time_current  filesize_current direction_current  \
0                      0.0               0.0               0.0   
10                     0.0               0.0               0.0   
57                     0.0               0.0               0.0   
105                    0.0               0.0               0.0   
2864                   0.0               0.0               0.0   
...                    ...               ...               ...   
17136562               0.0               0.0               0.0   
17140707               0.0               0.0               0.0   
17145467               0.0               0.0               0.0   
17149035               0.0               0.0               0.0   
17153702               0.0               0.0               0.0   

         timestamp_pw_idle  timetamp_ms_pw_idle  diff_pw_idle  \
0                      0.0                  0.0           0.0   
10                     0.0                  0.0           0.0   
57                     0.0                  0.0           0.0   
105                    0.0                  0.0           0.0   
2864                   0.0                  0.0           0.0   
...                    ...                  ...           ...   
17136562               0.0                  0.0           0.0   
17140707               0.0                  0.0           0.0   
17145467               0.0                  0.0           0.0   
17149035               0.0                  0.0           0.0   
17153702               0.0                  0.0           0.0   

          current_pw_idle  voltage_pw_idle  
0                     0.0              0.0  
10                    0.0              0.0  
57                    0.0              0.0  
105                   0.0              0.0  
2864                  0.0              0.0  
...                   ...              ...  
17136562              0.0              0.0  
17140707              0.0              0.0  
17145467              0.0              0.0  
17149035              0.0              0.0  
17153702              0.0              0.0  

[51060272 rows x 27 columns]
```

#### Output: preprocessed dataset reading
```small_test
Reading the content of /Users/username/NDA-Lab-Project4-RAT-Classification-TL/data/outcome_preprocess/feature_dataset.csv
/Users/username/NDA-Lab-Project4-RAT-Classification-TL/notebooks/../src/utils_train_models.py:71: DtypeWarning: Columns (14,15,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv(filepath)
Precomputed dataset:
             id  run node_name           location    modem_name  mcc  country  \
0         17443    1   Mark-10    Zagreb, Croatia  Quectel_BG96  219  Croatia   
1         17558    1   Mark-10    Zagreb, Croatia  Quectel_EC21  219  Croatia   
2         17760    1   Mark-10    Zagreb, Croatia  Quectel_BG96  219  Croatia   
3         17768    1   Mark-10    Zagreb, Croatia  Quectel_EC21  219  Croatia   
4         18830    1   Mark-10    Zagreb, Croatia  Quectel_EC21  219  Croatia   
...         ...  ...       ...                ...           ...  ...      ...   
51060267  23995    1    Mark-6    Munich, Germany  Quectel_BG96  262  Germany   
51060268  24001    1    Mark-1  Würzburg, Germany  Quectel_BG96  262  Germany   
51060269  24003    1    Mark-6    Munich, Germany  Quectel_BG96  262  Germany   
51060270  24009    1    Mark-1  Würzburg, Germany  Quectel_BG96  262  Germany   
51060271  24011    1    Mark-6    Munich, Germany  Quectel_BG96  262  Germany   

         iso_code  rat rat_name  ...  tot_size_current  avg_speed_current  \
0              hr    0       2G  ...               0.0                0.0   
1              hr    0       2G  ...               0.0                0.0   
2              hr    0       2G  ...               0.0                0.0   
3              hr    0       2G  ...               0.0                0.0   
4              hr    0       2G  ...               0.0                0.0   
...           ...  ...      ...  ...               ...                ...   
51060267       de    9   NB-IoT  ...               0.0                0.0   
51060268       de    9   NB-IoT  ...               0.0                0.0   
51060269       de    9   NB-IoT  ...               0.0                0.0   
51060270       de    9   NB-IoT  ...               0.0                0.0   
51060271       de    9   NB-IoT  ...               0.0                0.0   

          tot_time_current  filesize_current direction_current  \
0                      0.0               0.0               0.0   
1                      0.0               0.0               0.0   
2                      0.0               0.0               0.0   
3                      0.0               0.0               0.0   
4                      0.0               0.0               0.0   
...                    ...               ...               ...   
51060267               0.0               0.0               0.0   
51060268               0.0               0.0               0.0   
51060269               0.0               0.0               0.0   
51060270               0.0               0.0               0.0   
51060271               0.0               0.0               0.0   

         timestamp_pw_idle  timetamp_ms_pw_idle  diff_pw_idle  \
0                      0.0                  0.0           0.0   
1                      0.0                  0.0           0.0   
2                      0.0                  0.0           0.0   
3                      0.0                  0.0           0.0   
4                      0.0                  0.0           0.0   
...                    ...                  ...           ...   
51060267               0.0                  0.0           0.0   
51060268               0.0                  0.0           0.0   
51060269               0.0                  0.0           0.0   
51060270               0.0                  0.0           0.0   
51060271               0.0                  0.0           0.0   

          current_pw_idle  voltage_pw_idle  
0                     0.0              0.0  
1                     0.0              0.0  
2                     0.0              0.0  
3                     0.0              0.0  
4                     0.0              0.0  
...                   ...              ...  
51060267              0.0              0.0  
51060268              0.0              0.0  
51060269              0.0              0.0  
51060270              0.0              0.0  
51060271              0.0              0.0  

[51060272 rows x 27 columns]
```

##### Recover the correct column data type


In [ ]:
dataset = convert_to_numeric(dataset)

list_non_numerical = []
list_numerical = []
for col in dataset.columns:
    if pd.api.types.is_numeric_dtype(dataset[col]):
        list_numerical.append(col)
    else:
        list_non_numerical.append(col)

print("Numerical column list:\n")
print(list_numerical)
print()
print("Non numerical column list:\n")
print(list_non_numerical)

##### Output:<br>
Numerical column list:<br>
```small_test
['id', 'run', 'mcc', 'iso_code', 'rat', 'timestamp_throughput', 'tot_size_throughput', 'avg_speed_throughput', 'tot_time_throughput', 'timestamp_current', 'tot_size_current', 'avg_speed_current', 'tot_time_current', 'filesize_current', 'direction_current', 'timestamp_pw_idle', 'timetamp_ms_pw_idle', 'diff_pw_idle', 'current_pw_idle', 'voltage_pw_idle']
```

Non numerical column list:<br>
```small_test
['node_name', 'location', 'modem_name', 'country', 'rat_name', 'filesize_throughput', 'filesize_current', 'direction_throughput']
```

##### RAT_NAME saved as labels
This step is required for later evaluation of the trained model.<br>
It also simplifies the dataset structure for later partitioning of the preprocessed dataset into multiple subsections of fixed size.

In [ ]:
RAT_NAME = [label for label in np.unique(dataset["rat_name"])]
print("RAT classification, labels name: {}".format(RAT_NAME))
dataset.drop("rat_name", axis=1, inplace=True)

#####  Output:
```small_test
RAT classification, labels name: ["2G", "3G", "LTE CAT1", "LTE-M", "NB-IoT"]
```

#### Dataset encoding
To encode all values of type string among the most relevant non numeric columns of the original dataset.<br>
The process of measurement columns selection will filter the remaining non metadata information by ignoring non numeric columns (eg, throughput and current  direction).<br>

In [ ]:
dataset = encode_values(dataset)
dataset.to_csv(path_list["ENCODED_DATASET"], index=False)
print("Labels encoding process has been successfully completed and stored\n")
print("Encoded dataset obtained\n")
print(dataset)

##### Output:
```small_test
Column list with meaningful values of type string

['node_name', 'location', 'modem_name', 'country', 'filesize_throughput', 'direction_throughput']
Encoding string objects within the feature dataset...

Outcome 1 step encoding

   node_name  encoded
0     Mark-1        0
1    Mark-10        1
2    Mark-11        2
3    Mark-12        3
4     Mark-2        4
5     Mark-3        5
6     Mark-4        6
7     Mark-5        7
8     Mark-6        8
9     Mark-7        9
10    Mark-9       10
Column: node_name, Number of rows with NaN value: 11868601

Outcome 2 step encoding

            location  encoded
0       Milan, Italy        0
1    Munich, Germany        1
2  Trondheim, Norway        2
3  Würzburg, Germany        3
4    Zagreb, Croatia        4
Column: location, Number of rows with NaN value: 15631980

Outcome 3 step encoding

     modem_name  encoded
0  Quectel_BG96        0
1  Quectel_EC21        1
Column: modem_name, Number of rows with NaN value: 16006451

Outcome 4 step encoding

   country  encoded
0  Croatia        0
1  Germany        1
2    Italy        2
3   Norway        3
Column: country, Number of rows with NaN value: 15824230

Outcome 5 step encoding

  filesize_throughput  encoded
0                 0.0        0
1               100KB        1
2               200KB        2
3                 2MB        3
4               500KB        4
5                50KB        5
6                 5MB        6
Column: filesize_throughput, Number of rows with NaN value: 24238

Outcome 6 step encoding

  direction_throughput  encoded
0                  0.0        0
1             Downlink        1
2               Uplink        2
Column: direction_throughput, Number of rows with NaN value: 24238

Labels encoding process has been successfully completed

Encoded dataset obtained

             id  run  node_name  location  modem_name  mcc  country  iso_code  \
0         17443    1          1         4           0  219        0       0.0   
1         17558    1          1         4           1  219        0       0.0   
2         17760    1          1         4           0  219        0       0.0   
3         17768    1          1         4           1  219        0       0.0   
4         18830    1          1         4           1  219        0       0.0   
...         ...  ...        ...       ...         ...  ...      ...       ...   
51060267  23995    1          8         1           0  262        1       0.0   
51060268  24001    1          0         3           0  262        1       0.0   
51060269  24003    1          8         1           0  262        1       0.0   
51060270  24009    1          0         3           0  262        1       0.0   
51060271  24011    1          8         1           0  262        1       0.0   

          rat  timestamp_throughput  ...  tot_size_current  avg_speed_current  \
0           0                   0.0  ...               0.0                0.0   
1           0                   0.0  ...               0.0                0.0   
2           0                   0.0  ...               0.0                0.0   
3           0                   0.0  ...               0.0                0.0   
4           0                   0.0  ...               0.0                0.0   
...       ...                   ...  ...               ...                ...   
51060267    9                   0.0  ...               0.0                0.0   
51060268    9                   0.0  ...               0.0                0.0   
51060269    9                   0.0  ...               0.0                0.0   
51060270    9                   0.0  ...               0.0                0.0   
51060271    9                   0.0  ...               0.0                0.0   

          tot_time_current  filesize_current  direction_current  \
0                      0.0               0.0                0.0   
1                      0.0               0.0                0.0   
2                      0.0               0.0                0.0   
3                      0.0               0.0                0.0   
4                      0.0               0.0                0.0   
...                    ...               ...                ...   
51060267               0.0               0.0                0.0   
51060268               0.0               0.0                0.0   
51060269               0.0               0.0                0.0   
51060270               0.0               0.0                0.0   
51060271               0.0               0.0                0.0   

          timestamp_pw_idle  timetamp_ms_pw_idle  diff_pw_idle  \
0                       0.0                  0.0           0.0   
1                       0.0                  0.0           0.0   
2                       0.0                  0.0           0.0   
3                       0.0                  0.0           0.0   
4                       0.0                  0.0           0.0   
...                     ...                  ...           ...   
51060267                0.0                  0.0           0.0   
51060268                0.0                  0.0           0.0   
51060269                0.0                  0.0           0.0   
51060270                0.0                  0.0           0.0   
51060271                0.0                  0.0           0.0   

          current_pw_idle  voltage_pw_idle  
0                     0.0              0.0  
1                     0.0              0.0  
2                     0.0              0.0  
3                     0.0              0.0  
4                     0.0              0.0  
...                   ...              ...  
51060267              0.0              0.0  
51060268              0.0              0.0  
51060269              0.0              0.0  
51060270              0.0              0.0  
51060271              0.0              0.0  

[51060272 rows x 26 columns]
```

#### Generation of the windowed features dataset sorted by timestamp


In [ ]:
outcome_training_per_z = generate_outcome_training_per_z(dataset, RAT_NAME, "MLPClassifier")

##### Output:
```small_test
Measurement columns (8): ['tot_size_throughput', 'avg_speed_throughput', 'tot_time_throughput', 'filesize_throughput', 'direction_throughput', 'tot_size_current', 'avg_speed_current', 'tot_time_current']
Dataset was sorted by timestamps: ['timestamp_throughput', 'timestamp_current', 'timestamp_pw_idle']

Number of encoded groups: 85

Generation of the windowed dataset was ultimated
Train dataset length: 25717625, test dataset length: 25342647 obtained for Z: 1000
25718000it [02:48, 152958.06it/s]                              9.70it/s]
25343000it [02:52, 146979.80it/s]                              3.13it/s]
Windowed dataset computed for z = 1000

Length dataset: 51381

Training a NN...
Train dataset length: 25717625, test dataset length: 25342647 obtained for Z: 5000
25720000it [00:40, 628764.29it/s]                              8.70it/s]
25345000it [00:40, 630520.79it/s]                              0.08it/s]
Windowed dataset computed for z = 5000

Length dataset: 10453

Training a NN...

Train dataset length: 25717625, test dataset length: 25342647 obtained for Z: 10000
25720000it [00:25, 1023751.88it/s]                              6.40it/s]
25350000it [00:24, 1056040.59it/s]                              5.70it/s]
Windowed dataset computed for z = 10000

Length dataset: 5347

Training a NN...

Train dataset length: 25717625, test dataset length: 25342647 obtained for Z: 50000
25750000it [00:12, 2054658.35it/s]                              7.36it/s]
25350000it [00:13, 1941753.08it/s]                              1.37it/s]

Windowed dataset computed for z = 50000

Length dataset: 1261

Training a NN...
Train dataset length: 25717625, test dataset length: 25342647 obtained for Z: 100000
25800000it [00:11, 2180791.31it/s]                              3.93it/s]
25400000it [00:11, 2245797.63it/s]                              5.56it/s]

Windowed dataset computed for z = 100000

Length dataset: 751

Training a NN...
Train dataset length: 25717625, test dataset length: 25342647 obtained for Z: 500000
26000000it [00:10, 2527240.45it/s]                              8.46it/s]
25500000it [00:10, 2489300.37it/s]                              6.81it/s]

Windowed dataset computed for z = 500000

Length dataset: 337

Training a NN...
```

#### Statistics computation

In [ ]:
accuracy_per_z = compute_statistics(outcome_training_per_z, RAT_NAME)

##### Example output 
This outcome refers to a previous code execution <br>
```small_test
------------------------------------

Results for Z: 1000

Training time[s]: 5.748613119125366

Accuracy: 0.904292751583392

Global precision: 0.9324413085954311

Global recall: 0.904292751583392

Global f1score: 0.9099170614918106

------------------------------------

------------------------------------

Results for Z: 5000

Training time[s]: 1.3329339027404785

Accuracy: 0.8891653905053599

Global precision: 0.9201880857718344

Global recall: 0.8891653905053599

Global f1score: 0.8937860135272397

------------------------------------

------------------------------------

Results for Z: 10000

Training time[s]: 0.665402889251709

Accuracy: 0.9044609665427509

Global precision: 0.927983191973203

Global recall: 0.9044609665427509

Global f1score: 0.911556126173508

------------------------------------

------------------------------------

Results for Z: 50000

Training time[s]: 0.15201902389526367

Accuracy: 0.6631419939577039

Global precision: 0.4898169779549967

Global recall: 0.6631419939577039

Global f1score: 0.5632197750533301

------------------------------------

------------------------------------

Results for Z: 100000

Training time[s]: 0.09532308578491211

Accuracy: 0.6185819070904646

Global precision: 0.4794509791947026

Global recall: 0.6185819070904646

Global f1score: 0.52084956608184

------------------------------------

------------------------------------

Results for Z: 500000

Training time[s]: 0.0466001033782959

Accuracy: 0.49504950495049505

Global precision: 0.39957265996869956

Global recall: 0.49504950495049505

Global f1score: 0.39342977341869106

------------------------------------
```

#### Selection of the pretrained model with highest accuracy
To select and export both the windowed dataset associated to the pretrained model with the highest accuracy and the pretrained model itself.

In [9]:
max_accuracy = max(accuracy_per_z)
outcome_windowed_dataset_eval = {}
for index_row, accuracy in enumerate(accuracy_per_z):
    if accuracy == max_accuracy:
        outcome_windowed_dataset_eval["window_size"] = outcome_training_per_z.loc[index_row]["z_value"]
        outcome_windowed_dataset_eval["selected_model"] = outcome_training_per_z.loc[index_row]["pretrained_model"]
        outcome_windowed_dataset_eval["windowed_dataset"] = outcome_training_per_z.loc[index_row]["features_set"]
        break
    
store_windowed_training_analysis(outcome_windowed_dataset_eval, MODEL_NAME)

##### Output
```small_text
--------------------------------------------------------------------------
MOST ACCURATE WINDOWED MODEL
Window size:1000
Windowed dataset size: 40
Windowed dataset length: 51381
Windowed dataset columns:  "run","node_name","location","modem_name","mcc","country","iso_code","rat",
            "tot_size_throughput_min",tot_size_throughput_max","tot_size_throughput_std","tot_size_throughput_mean",
            "avg_speed_throughput_min","avg_speed_throughput_max","avg_speed_throughput_std","avg_speed_throughput_mean","tot_time_throughput_min","tot_time_throughput_max","tot_time_throughput_std","tot_time_throughput_mean","filesize_throughput_min","filesize_throughput_max","filesize_throughput_std",
            "filesize_throughput_mean","direction_throughput_min","direction_throughput_max","direction_throughput_std","direction_throughput_mean",
            "tot_size_current_min","tot_size_current_max","tot_size_current_std","tot_size_current_mean",
            "avg_speed_current_min","avg_speed_current_max","avg_speed_current_std","avg_speed_current_mean",
            "tot_time_current_min","tot_time_current_max","tot_time_current_std","tot_time_current_mean"
--------------------------------------------------------------------------
```

#### To export the selected pretrained model in ONNX format
This step could be useful for visualizing all modules included in the ONNX graph of the pretrained model.<br>
There are some Python libraries, such as *onnx2torch* or *onnx2keras*, that aim to convert ONNX modules in their corresponding PyTorch or Keras ones.<br>
This will provide a useful insight for model adaptations through fine tuning in the Transfer Learning task.

In [10]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# To consider all columns except the target one
input_dim = outcome_windowed_dataset_eval["windowed_dataset"].shape[1] - 1
initial_type = [("float_input", FloatTensorType([None, input_dim]))]

onx = convert_sklearn(outcome_windowed_dataset_eval["selected_model"], initial_types = initial_type)

with open(os.path.join(path_list["EXPORTED_DIR"],MODEL_NAME+".onnx"),"wb") as f:
    f.write(onx.SerializeToString())

##### Output
<p align="center">
<img src="../results/exported/MLPClassifier.png" alt="Pretrained model graph" width="400">
</p>